In [ ]:
!pip uninstall qdrant-client -y
!pip install qdrant-client==1.9.1

In [ ]:
pip install -U qdrant-client sentence-transformers pymupdf

In [2]:
!pip install -U langchain
# Requires Python 3.10+

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 2.6 MB/s eta 0:00:00


In [15]:
import os
import fitz
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.http.models import (
    VectorParams,
    Distance,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue
)
from qdrant_client.models import PayloadSchemaType

# ==========================
# LOAD ENV VARIABLES
# ==========================

load_dotenv()

QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")

# ==========================
# CONNECT QDRANT
# ==========================

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
    check_compatibility=False
)

# ==========================
# LOAD PDF
# ==========================

pdf = fitz.open("50_page_product_catalog.pdf")

# ==========================
# EMBEDDING MODEL
# ==========================

model = SentenceTransformer("all-MiniLM-L6-v2")

collection_name = "clothing_collections"

# ==========================
# DELETE OLD COLLECTION
# ==========================

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

# ==========================
# CREATE COLLECTION
# ==========================

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

# ==========================
# CREATE PAYLOAD INDEXES
# ==========================

for field in ["Color", "Brand", "Season", "Name"]:
    client.create_payload_index(
        collection_name=collection_name,
        field_name=field,
        field_schema=PayloadSchemaType.KEYWORD
    )

print("✅ Payload indexes created")

# ==========================
# EXTRACT FIELDS
# ==========================

def extract_fields(text):

    lines = [line.strip() for line in text.split("\n") if line.strip()]

    valid_fields = [
        "Name",
        "Brand",
        "Price",
        "Color",
        "Sizes",
        "Season",
        "Stock",
        "Minimum Age",
        "Description"
    ]

    data = {}

    for i in range(len(lines) - 1):

        if lines[i] in valid_fields:
            data[lines[i]] = lines[i + 1]

    return data

# ==========================
# PREPARE POINTS
# ==========================

points = []

for page_num in range(len(pdf)):

    page = pdf[page_num]

    text = page.get_text().strip()

    fields = extract_fields(text)

    embedding = model.encode(text)

    payload = {
        "page": page_num + 1,
        "text": text,
        **fields
    }

    point = PointStruct(
        id=page_num,
        vector=embedding.tolist(),
        payload=payload
    )

    points.append(point)

# ==========================
# UPSERT DATA
# ==========================

client.upsert(
    collection_name=collection_name,
    points=points
)

print("✅ Data stored successfully")

# ==========================
# VERIFY DATA
# ==========================

records, _ = client.scroll(
    collection_name=collection_name,
    limit=3,
    with_payload=True
)

print("\nSample Records:\n")

for record in records:
    print(record.payload)
    print()

# ==========================
# SEARCH QUERY
# ==========================

query = "black shirt"

query_embedding = model.encode(query)

# ==========================
# FILTERED SEARCH
# ==========================

search_results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(
                key="Color",
                match=MatchValue(value="Black")
            ),
            FieldCondition(
            key="Name",
            match=MatchValue(value="Black Shirt")
            )
        ]
    ),
    limit=3
).points

# ==========================
# DISPLAY RESULTS
# ==========================

print("\n🔍 SEARCH RESULTS\n")

for res in search_results:

    print("Score:", round(res.score, 3))
    print("Page:", res.payload.get("page"))
    print("Name:", res.payload.get("Name"))
    print("Brand:", res.payload.get("Brand"))
    print("Price:", res.payload.get("Price"))
    print("Color:", res.payload.get("Color"))
    print("Sizes:", res.payload.get("Sizes"))
    print("Season:", res.payload.get("Season"))
    print("Stock:", res.payload.get("Stock"))
    print("Minimum Age:", res.payload.get("Minimum Age"))
    print("Description:", res.payload.get("Description"))

    print("-" * 60)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Payload indexes created
✅ Data stored successfully

Sample Records:

{'page': 1, 'text': 'Product Catalog Page 1\nField\nValue\nName\nBlack Shirt\nBrand\nPuma\nPrice\nI1968\nColor\nBlack\nSizes\nS, XL\nSeason\nSummer\nStock\n27\nMinimum Age\n18\nDescription\nPremium quality t-shirt made with comfortable fabric.', 'Name': 'Black Shirt', 'Brand': 'Puma', 'Price': 'I1968', 'Color': 'Black', 'Sizes': 'S, XL', 'Season': 'Summer', 'Stock': '27', 'Minimum Age': '18', 'Description': 'Premium quality t-shirt made with comfortable fabric.'}

{'page': 2, 'text': 'Product Catalog Page 2\nField\nValue\nName\nBlack Shirt\nBrand\nNike\nPrice\nI1761\nColor\nRed\nSizes\nL, XL, M, S\nSeason\nAll Season\nStock\n38\nMinimum Age\n18\nDescription\nPremium quality jacket made with comfortable fabric.', 'Name': 'Black Shirt', 'Brand': 'Nike', 'Price': 'I1761', 'Color': 'Red', 'Sizes': 'L, XL, M, S', 'Season': 'All Season', 'Stock': '38', 'Minimum Age': '18', 'Description': 'Premium quality jacket made with 